# Nettoyage et enrichissement des données TGV + météo

Ce notebook reprend la logique du notebook d'origine en la rendant plus lisible :

1. chargement des données,
2. nettoyage et appariement des gares,
3. préparation des coordonnées,
4. récupération des données météo mensuelles,
5. enrichissement géographique (commune, département, région),
6. exports.

Les cellules d'exploration ponctuelle et les essais intermédiaires ont été retirés pour garder un flux de travail unique.

In [1]:
from pathlib import Path
import os
import re
import time
import unicodedata

import pandas as pd
import requests_cache
import openmeteo_requests
from retry_requests import retry

## Configuration

In [2]:
DATA_DIR = Path(".")

FILES = {
    "regularite": DATA_DIR / "regularite-mensuelle-tgv-aqst.csv",
    "gares": DATA_DIR / "gares-de-voyageurs.csv",
    "communes": DATA_DIR / "v_commune_2026.csv",
    "departements": DATA_DIR / "departements-france.csv",
}

SAVE_DIR = DATA_DIR / "weather_batches"
SAVE_DIR.mkdir(exist_ok=True)

DAILY_VARS = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",
    "wind_speed_10m_max",
    "wind_gusts_10m_max",
    "daylight_duration",
]

GARES_ETRANGERES = [
    "BARCELONA", "FRANCFORT", "GENEVE", "ITALIE",
    "LAUSANNE", "MADRID", "STUTTGART", "ZURICH",
]

CORRECTIONS_GARES = {
    "BELLEGARDE (AIN)": "Bellegarde-sur-Valserine",
    "BORDEAUX ST JEAN": "Bordeaux Saint-Jean",
    "DIJON VILLE": "Dijon",
    "LA ROCHELLE VILLE": "La Rochelle",
    "LILLE": "Lille Europe",
    "MACON LOCHE": "Mâcon Loché TGV",
    "MARNE LA VALLEE": "Marne-la-Vallée Chessy",
    "MARSEILLE ST CHARLES": "Marseille Saint-Charles",
    "MONTPELLIER": "Montpellier Sud de France",
    "MULHOUSE VILLE": "Mulhouse",
    "NICE VILLE": "Nice",
    "PARIS LYON": "Paris Gare de Lyon",
    "ST MALO": "Saint-Malo",
    "ST PIERRE DES CORPS": "Saint-Pierre-des-Corps",
    "VALENCE ALIXAN TGV": "Valence TGV Rhône-Alpes Sud",
    "LE CREUSOT MONTCEAU MONTCHANIN": "LE CREUSOT MONTCEAU LES MINES MONTCHANIN TGV",
    "PARIS NORD": "PARIS GARE DU NORD",
}

## Fonctions utilitaires

In [3]:
def clean_gare(name: str) -> str:
    """Normalise un nom de gare pour faciliter l'appariement."""
    if pd.isna(name):
        return name

    name = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode("utf-8")
    name = name.upper()
    name = re.sub(r"[^A-Z0-9 ]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def normalize_code(series: pd.Series, width: int) -> pd.Series:
    """Convertit une série en code texte propre en conservant les zéros à gauche."""
    return (
        series.astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
        .str.zfill(width)
    )


def add_lat_lon_from_position(df: pd.DataFrame, position_col: str = "Position géographique") -> pd.DataFrame:
    """Extrait latitude et longitude depuis une colonne 'lat,lon'."""
    out = df.copy()
    coords = out[position_col].str.split(",", expand=True)
    out["latitude"] = pd.to_numeric(coords[0], errors="coerce")
    out["longitude"] = pd.to_numeric(coords[1], errors="coerce")
    return out


def build_station_reference(df_regularite: pd.DataFrame, df_gares: pd.DataFrame) -> pd.DataFrame:
    """Construit la table de correspondance gare d'origine -> gare référentielle + coordonnées."""
    gares_depart = pd.DataFrame(
        {"Gare de départ": sorted(df_regularite["Gare de départ"].dropna().unique())}
    )

    gares_depart = gares_depart[
        ~gares_depart["Gare de départ"].astype(str).str.isnumeric()
    ].copy()

    gares_depart = gares_depart[
        ~gares_depart["Gare de départ"].isin(GARES_ETRANGERES)
    ].copy()

    gares_depart["Gare_match"] = gares_depart["Gare de départ"].replace(CORRECTIONS_GARES)
    gares_depart["gare_clean"] = gares_depart["Gare_match"].apply(clean_gare)

    gares_ref = df_gares.copy()
    gares_ref["gare_clean"] = gares_ref["Nom_Gare"].apply(clean_gare)

    merged = gares_depart.merge(
        gares_ref[["Nom_Gare", "gare_clean", "Position géographique", "Code commune"]],
        on="gare_clean",
        how="left",
    )

    merged = add_lat_lon_from_position(merged)
    return merged


def create_openmeteo_client():
    cache_session = requests_cache.CachedSession(".cache", expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    return openmeteo_requests.Client(session=retry_session)


def get_monthly_weather_batch_resume(
    df_stations: pd.DataFrame,
    start_date: str,
    end_date: str,
    daily_vars: list[str],
    openmeteo_client,
    batch_size: int = 10,
    pause: int = 65,
    start_batch: int = 1,
    save_dir: str | Path = "weather_batches",
) -> pd.DataFrame:
    """Récupère les données météo Open-Meteo par lots et sauvegarde chaque lot en CSV."""
    save_dir = Path(save_dir)
    save_dir.mkdir(exist_ok=True)

    url = "https://archive-api.open-meteo.com/v1/archive"
    results = []

    stations = (
        df_stations[["Gare de départ", "latitude", "longitude"]]
        .drop_duplicates()
        .dropna(subset=["latitude", "longitude"])
        .reset_index(drop=True)
    )

    total_batches = (len(stations) + batch_size - 1) // batch_size
    print(f"Nombre total de batches : {total_batches}")

    for batch_num, start_idx in enumerate(range(0, len(stations), batch_size), start=1):
        if batch_num < start_batch:
            continue

        batch = stations.iloc[start_idx:start_idx + batch_size].copy()
        output_file = save_dir / f"batch_{batch_num}.csv"

        if output_file.exists():
            print(f"Batch {batch_num} déjà présent, ignoré.")
            continue

        params = {
            "latitude": batch["latitude"].tolist(),
            "longitude": batch["longitude"].tolist(),
            "start_date": start_date,
            "end_date": end_date,
            "daily": daily_vars,
            "timezone": "Europe/Paris",
        }

        try:
            responses = openmeteo_client.weather_api(url, params=params)
            batch_results = []

            for station_name, response in zip(batch["Gare de départ"], responses):
                daily = response.Daily()
                daily_data = {
                    "date": pd.date_range(
                        start=pd.to_datetime(
                            daily.Time() + response.UtcOffsetSeconds(), unit="s", utc=True
                        ),
                        end=pd.to_datetime(
                            daily.TimeEnd() + response.UtcOffsetSeconds(), unit="s", utc=True
                        ),
                        freq=pd.Timedelta(seconds=daily.Interval()),
                        inclusive="left",
                    )
                }

                for idx, var in enumerate(daily_vars):
                    daily_data[var] = daily.Variables(idx).ValuesAsNumpy()

                df_daily = pd.DataFrame(daily_data)
                df_daily["date"] = pd.to_datetime(df_daily["date"]).dt.tz_localize(None)
                df_daily["Date"] = df_daily["date"].dt.to_period("M").dt.to_timestamp()
                df_daily["Gare de départ"] = station_name

                df_monthly = (
                    df_daily.groupby(["Gare de départ", "Date"], as_index=False)
                    .agg({
                        "temperature_2m_mean": "mean",
                        "temperature_2m_max": "max",
                        "temperature_2m_min": "min",
                        "precipitation_sum": "sum",
                        "rain_sum": "sum",
                        "snowfall_sum": "sum",
                        "wind_speed_10m_max": "max",
                        "wind_gusts_10m_max": "max",
                        "daylight_duration": "mean",
                    })
                )
                batch_results.append(df_monthly)

            df_batch = pd.concat(batch_results, ignore_index=True)
            df_batch.to_csv(output_file, index=False)
            results.append(df_batch)

            print(f"Batch {batch_num}/{total_batches} sauvegardé : {output_file}")

            if batch_num < total_batches:
                time.sleep(pause)

        except Exception as exc:
            print(f"Erreur sur le batch {batch_num}: {exc}")
            print(f"Relancer avec start_batch={batch_num}")
            break

    if results:
        return pd.concat(results, ignore_index=True)

    return pd.DataFrame()


def combine_weather_batches(save_dir: str | Path = "weather_batches") -> pd.DataFrame:
    """Concatène tous les lots météo sauvegardés dans un dossier."""
    save_dir = Path(save_dir)
    files = sorted(save_dir.glob("batch_*.csv"))

    if not files:
        return pd.DataFrame()

    weather = pd.concat((pd.read_csv(file) for file in files), ignore_index=True)
    weather["Date"] = pd.to_datetime(weather["Date"])
    return weather

## Chargement des données sources

In [4]:
df_regularite = pd.read_csv(FILES["regularite"], sep=";")
df_gares = pd.read_csv(FILES["gares"], sep=";")
df_communes = pd.read_csv(FILES["communes"])
df_departements = pd.read_csv(FILES["departements"])

print(df_regularite.shape)
print(df_gares.shape)
print(df_communes.shape)
print(df_departements.shape)

(11834, 26)
(2782, 7)
(37496, 12)
(101, 4)


## Préparation des gares et des coordonnées

In [5]:
df_station_reference = build_station_reference(df_regularite, df_gares)

df_stations_clean = (
    df_station_reference[["Gare de départ", "latitude", "longitude"]]
    .drop_duplicates()
    .dropna(subset=["latitude", "longitude"])
    .reset_index(drop=True)
)

display(df_station_reference.head())
print("Nombre de gares avec coordonnées :", len(df_stations_clean))

,Gare de départ,Gare_match,gare_clean,Nom_Gare,Position géographique,Code commune,latitude,longitude
0,AIX EN PROVENCE TGV,AIX EN PROVENCE TGV,AIX EN PROVENCE TGV,Aix-en-Provence TGV,"43.455237, 5.317534",13001.0,43.455237,5.317534
1,ANGERS SAINT LAUD,ANGERS SAINT LAUD,ANGERS SAINT LAUD,Angers Saint-Laud,"47.464647, -0.55682",49007.0,47.464647,-0.556820
2,ANGOULEME,ANGOULEME,ANGOULEME,Angoulême,"45.653572, 0.164608",16015.0,45.653572,0.164608
3,ANNECY,ANNECY,ANNECY,Annecy,"45.901965, 6.121835",74010.0,45.901965,6.121835
4,ARRAS,ARRAS,ARRAS,Arras,"50.286673, 2.781942",62041.0,50.286673,2.781942


Nombre de gares avec coordonnées : 50


## Période météo à couvrir

In [6]:
df_regularite["Date"] = pd.to_datetime(df_regularite["Date"])

start_date = df_regularite["Date"].min().strftime("%Y-%m-%d")
end_date = (df_regularite["Date"].max() + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")

print("start_date =", start_date)
print("end_date   =", end_date)

start_date = 2018-01-01
end_date   = 2025-12-31


## Client Open-Meteo

Décommente la cellule suivante uniquement si tu veux relancer la collecte API.

In [7]:
openmeteo = create_openmeteo_client()

In [8]:
# Exemple d'exécution :
# df_weather_part = get_monthly_weather_batch_resume(
#     df_stations=df_stations_clean,
#     start_date=start_date,
#     end_date=end_date,
#     daily_vars=DAILY_VARS,
#     openmeteo_client=openmeteo,
#     batch_size=10,
#     pause=65,
#     start_batch=1,
#     save_dir=SAVE_DIR,
# )

## Chargement des batches météo déjà sauvegardés

In [9]:
df_weather_monthly = combine_weather_batches(SAVE_DIR)

print(df_weather_monthly.shape)
display(df_weather_monthly.head())

(4800, 11)


,Gare de départ,Date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,wind_gusts_10m_max,daylight_duration
0,AIX EN PROVENCE TGV,2018-01-01,9.691331,17.55,1.7,60.899998,60.899998,0.000000,44.645800,83.520004,33768.734
1,AIX EN PROVENCE TGV,2018-02-01,4.706250,14.45,-3.7,42.300003,33.400000,6.229999,48.828793,90.720000,37842.780
2,AIX EN PROVENCE TGV,2018-03-01,9.380779,18.05,-0.9,77.100000,73.600000,2.450000,52.260790,98.279990,42985.414
3,AIX EN PROVENCE TGV,2018-04-01,14.830277,25.70,5.5,81.700000,81.700000,0.000000,42.696060,89.640000,48359.400
4,AIX EN PROVENCE TGV,2018-05-01,17.180914,28.75,6.9,104.300000,104.300000,0.000000,38.183765,71.640000,52941.720


## Enrichissement géographique : commune, département, région

In [16]:
df_geo = (
    df_station_reference[
        ["Gare de départ", "Gare_match", "Nom_Gare", "latitude", "longitude", "Code commune"]
    ]
    .drop_duplicates()
    .copy()
)

df_geo["Code commune"] = normalize_code(df_geo["Code commune"], width=5)

df_communes_ref = df_communes[["COM", "DEP", "REG"]].copy()
df_communes_ref["COM"] = normalize_code(df_communes_ref["COM"], width=5)
df_communes_ref["DEP"] = normalize_code(df_communes_ref["DEP"], width=2)
df_communes_ref["REG"] = normalize_code(df_communes_ref["REG"], width=2)

# On enlève d'abord les lignes totalement inutiles / incomplètes
df_communes_ref = df_communes_ref.dropna(subset=["COM"])

# On privilégie les lignes les plus complètes
df_communes_ref = (
    df_communes_ref
    .assign(nb_na=df_communes_ref[["DEP", "REG"]].isna().sum(axis=1))
    .sort_values(["COM", "nb_na"])
    .drop_duplicates(subset=["COM"], keep="first")
    .drop(columns="nb_na")
)

df_departements_ref = df_departements.copy()
df_departements_ref["code_departement"] = normalize_code(
    df_departements_ref["code_departement"], width=2
)

df_departements_ref = (
    df_departements_ref
    .dropna(subset=["code_departement"])
    .drop_duplicates(subset=["code_departement"])
)

df_info_geo = (
    df_geo
    .merge(
        df_communes_ref,
        left_on="Code commune",
        right_on="COM",
        how="left",
        validate="many_to_one"
    )
    .merge(
        df_departements_ref,
        left_on="DEP",
        right_on="code_departement",
        how="left",
        validate="many_to_one"
    )
    .drop(columns=["COM"], errors="ignore")
)

print(df_info_geo.shape)
display(df_info_geo.head())

(51, 12)


,Gare de départ,Gare_match,Nom_Gare,latitude,longitude,Code commune,DEP,REG,code_departement,nom_departement,code_region,nom_region
0,AIX EN PROVENCE TGV,AIX EN PROVENCE TGV,Aix-en-Provence TGV,43.455237,5.317534,13001,13,93,13,Bouches-du-Rhône,93.0,Provence-Alpes-Côte d'Azur
1,ANGERS SAINT LAUD,ANGERS SAINT LAUD,Angers Saint-Laud,47.464647,-0.556820,49007,49,52,49,Maine-et-Loire,52.0,Pays de la Loire
2,ANGOULEME,ANGOULEME,Angoulême,45.653572,0.164608,16015,16,75,16,Charente,75.0,Nouvelle-Aquitaine
3,ANNECY,ANNECY,Annecy,45.901965,6.121835,74010,74,84,74,Haute-Savoie,84.0,Auvergne-Rhône-Alpes
4,ARRAS,ARRAS,Arras,50.286673,2.781942,62041,62,32,62,Pas-de-Calais,32.0,Hauts-de-France


## Contrôles qualité rapides

In [17]:
print("Valeurs manquantes par colonne :")
display(df_info_geo.isna().sum().sort_values(ascending=False))

print("\nGares sans coordonnées :")
display(df_station_reference[df_station_reference["latitude"].isna() | df_station_reference["longitude"].isna()])

Valeurs manquantes par colonne :


Nom_Gare            1
latitude            1
longitude           1
Code commune        1
DEP                 1
REG                 1
code_departement    1
nom_departement     1
code_region         1
nom_region          1
Gare de départ      0
Gare_match          0
dtype: int64


Gares sans coordonnées :


,Gare de départ,Gare_match,gare_clean,Nom_Gare,Position géographique,Code commune,latitude,longitude
35,PARIS VAUGIRARD,PARIS VAUGIRARD,PARIS VAUGIRARD,NaN,NaN,NaN,NaN,NaN


In [20]:
df_info_geo =  df_info_geo.dropna()

print("Valeurs manquantes par colonne :")
display(df_info_geo.isna().sum().sort_values(ascending=False))



Valeurs manquantes par colonne :


Gare de départ      0
Gare_match          0
Nom_Gare            0
latitude            0
longitude           0
Code commune        0
DEP                 0
REG                 0
code_departement    0
nom_departement     0
code_region         0
nom_region          0
dtype: int64

## Exports

In [12]:
df_stations_clean.to_csv("df_stations_clean.csv", index=False)
df_weather_monthly.to_csv("df_weather_monthly.csv", index=False)
df_info_geo.to_csv("df_info_geo.csv", index=False)

print("Fichiers exportés : df_stations_clean.csv, df_weather_monthly.csv, df_info_geo.csv")

Fichiers exportés : df_stations_clean.csv, df_weather_monthly.csv, df_info_geo.csv
